In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import geopandas as gpd

from src.cleaners import sanitize_listings_pipeline
from src.schemas import CleanedListingsSchema
from src.spatial import compute_transit_proximity, spatial_join_neighborhoods
from src.imputers import run_imputation_pipeline

pd.set_option('display.max_columns', 50)

In [ ]:
# Step 1: Load raw data and sanitize
raw_path = "../data/raw/listings.csv.gz"
df_raw = pd.read_csv(raw_path, compression='gzip', low_memory=False)
print(f"Raw dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

df_clean = sanitize_listings_pipeline(df_raw)
df_clean = CleanedListingsSchema.validate(df_clean)
print(f"Sanitized dataset: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")
df_clean.head(3)

In [ ]:
# Step 2: Geospatial Harmonization with Subway & Neighborhood Polygons
df_subway = pd.read_csv("../data/raw/subway_stations.csv")
df_spatial = compute_transit_proximity(df_clean, df_subway, radius_km=1.0)

geojson_path = "../data/raw/neighbourhoods.geojson"
df_spatial = spatial_join_neighborhoods(df_spatial, geojson_path)

print(f"Spatial features added: {df_spatial.shape}")
df_spatial[['neighbourhood_cleansed', 'distance_to_nearest_subway_km', 'subway_count_1km']].head(3)

In [ ]:
# Step 3: Missing Value Imputation
print("Missing counts before imputation:")
print(df_spatial[['bedrooms', 'beds', 'bathrooms', 'review_scores_rating']].isna().sum())

df_imputed = run_imputation_pipeline(df_spatial)

print("\nMissing counts after imputation:")
print(df_imputed[['bedrooms', 'beds', 'bathrooms', 'review_scores_rating']].isna().sum())

In [ ]:
# Step 4: Outlier Treatment & Feature Store Generation
df_model = df_imputed[df_imputed['price'].notna()].copy()
df_model = df_model[(df_model['price'] >= 20) & (df_model['price'] <= 2500)]

df_model['log_price'] = np.log1p(df_model['price'])
df_model['price_per_bedroom'] = np.round(df_model['price'] / (df_model['bedrooms'] + 0.5), 2)

processed_path = "../data/processed/feature_store.parquet"
df_model.drop(columns=['amenities_list']).to_parquet(processed_path, index=False)
print(f"Saved final feature store: {df_model.shape} to {processed_path}")
df_model.head(3)